In [2]:
# ============================================================
# 1. INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install -q transformers datasets accelerate scikit-learn pandas openpyxl tqdm


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import os
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report


# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# ============================================================
# 4. LOAD DATASET
# ============================================================

# Change this path according to your dataset file
DATA_PATH = "/content/5K_HateSpeechAllGender.xlsx"

df = pd.read_excel(DATA_PATH)

print(df.head())
print(df.columns)


# ============================================================
# 5. COLUMN CONFIGURATION
# ============================================================

# Change these column names according to your actual dataset
TEXT_COL = df.columns[0]   # first column as text column

LABEL_COLS = [
    "Level1 - CLASS", # Corrected from "Level 1 - Class"
    "Level 2 - TARGET",
    "Level 3 - CATEGORIES", # Corrected from "Level 3 - Category"
    "Level 4 - SUBCATEGORIES", # Corrected from "Level 4 - Subcategories"
    "Level 5 - BIAS"
]

# If your dataset column names are different, use like this:
# LABEL_COLS = ["Class", "Target", "Category", "Subcategory", "Bias"]

# Fill missing labels
df[TEXT_COL] = df[TEXT_COL].astype(str)
for col in LABEL_COLS:
    df[col] = df[col].fillna("Nil").astype(str)

print("Text column:", TEXT_COL)
print("Label columns:", LABEL_COLS)


# ============================================================
# 6. LABEL ENCODING
# ============================================================

label_encoders = {}
num_labels_dict = {}

for col in LABEL_COLS:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    num_labels_dict[col] = len(le.classes_)
    print(f"\n{col}")
    print("Classes:", list(le.classes_))
    print("Number of classes:", num_labels_dict[col])


# ============================================================
# 7. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

# Stratify using Level 1 if possible
stratify_col = LABEL_COLS[0]

try:
    train_df, temp_df = train_test_split(
        df,
        test_size=0.20,
        random_state=42,
        stratify=df[stratify_col]
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=42,
        stratify=temp_df[stratify_col]
    )

except:
    print("Stratified split failed due to rare classes. Using random split.")

    train_df, temp_df = train_test_split(
        df,
        test_size=0.20,
        random_state=42
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=42
    )

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


# ============================================================
# 8. DATASET CLASS
# ============================================================

class HierarchicalHateDataset(Dataset):
    def __init__(self, dataframe, tokenizer, text_col, label_cols, max_len=128):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.text_col = text_col
        self.label_cols = label_cols
        self.max_len = max_len

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        text = str(row[self.text_col])

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )

        labels = torch.tensor(
            [row[col] for col in self.label_cols],
            dtype=torch.long
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": labels
        }


# ============================================================
# 9. MULTI-HEAD TRANSFORMER MODEL
# ============================================================

class MultiHeadTransformerClassifier(nn.Module):
    def __init__(self, model_name, num_labels_list, dropout=0.3):
        super(MultiHeadTransformerClassifier, self).__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)

        self.classifiers = nn.ModuleList([
            nn.Linear(hidden_size, num_labels)
            for num_labels in num_labels_list
        ])

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Some models provide pooler_output, some do not
        if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            pooled_output = outputs.pooler_output
        else:
            pooled_output = outputs.last_hidden_state[:, 0, :]

        pooled_output = self.dropout(pooled_output)

        logits = [classifier(pooled_output) for classifier in self.classifiers]

        return logits


# ============================================================
# 10. CLASS WEIGHTS FOR IMBALANCED DATA
# ============================================================

def compute_class_weights(train_df, label_cols, num_labels_dict):
    class_weights = []

    for col in label_cols:
        counts = train_df[col].value_counts().sort_index()
        total = counts.sum()
        num_classes = num_labels_dict[col]

        weights = []
        for i in range(num_classes):
            count = counts.get(i, 0)
            if count == 0:
                weights.append(0.0)
            else:
                weights.append(total / (num_classes * count))

        weights = torch.tensor(weights, dtype=torch.float).to(device)
        class_weights.append(weights)

    return class_weights


# ============================================================
# 11. TRAINING FUNCTION
# ============================================================

def train_one_epoch(model, dataloader, optimizer, scheduler, loss_fns):
    model.train()

    total_loss = 0

    for batch in tqdm(dataloader, desc="Training", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits_list = model(input_ids, attention_mask)

        loss = 0
        for i, logits in enumerate(logits_list):
            loss += loss_fns[i](logits, labels[:, i])

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


# ============================================================
# 12. EVALUATION FUNCTION
# ============================================================

def evaluate_model(model, dataloader, loss_fns, label_cols):
    model.eval()

    total_loss = 0

    all_true = [[] for _ in label_cols]
    all_pred = [[] for _ in label_cols]

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits_list = model(input_ids, attention_mask)

            loss = 0
            for i, logits in enumerate(logits_list):
                loss += loss_fns[i](logits, labels[:, i])

                preds = torch.argmax(logits, dim=1)

                all_true[i].extend(labels[:, i].cpu().numpy())
                all_pred[i].extend(preds.cpu().numpy())

            total_loss += loss.item()

    results = {}
    level_f1_scores = []

    for i, col in enumerate(label_cols):
        acc = accuracy_score(all_true[i], all_pred[i])

        precision, recall, f1, _ = precision_recall_fscore_support(
            all_true[i],
            all_pred[i],
            average="macro",
            zero_division=0
        )

        results[col] = {
            "accuracy": acc,
            "macro_precision": precision,
            "macro_recall": recall,
            "macro_f1": f1
        }

        level_f1_scores.append(f1)

    results["average_macro_f1"] = np.mean(level_f1_scores)
    results["loss"] = total_loss / len(dataloader)

    return results, all_true, all_pred


# ============================================================
# 13. BASELINE MODELS
# ============================================================

baseline_models = {
    "mBERT": "bert-base-multilingual-cased",
    "XLM-RoBERTa": "xlm-roberta-base",
    "DistilBERT": "distilbert-base-multilingual-cased",
    "IndicBERT": "ai4bharat/indic-bert",
    "MuRIL": "google/muril-base-cased",
    "ALBERT": "albert-base-v2",
    "MiniLM": "microsoft/Multilingual-MiniLM-L12-H384",
    "RoBERTa": "roberta-base"
}


# ============================================================
# 14. TRAIN ALL BASELINE MODELS
# ============================================================

MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 2e-5

num_labels_list = [num_labels_dict[col] for col in LABEL_COLS]

all_model_results = {}

for model_display_name, model_name in baseline_models.items():

    print("\n" + "="*80)
    print(f"Training baseline model: {model_display_name}")
    print(f"HuggingFace model: {model_name}")
    print("="*80)

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)

        train_dataset = HierarchicalHateDataset(
            train_df, tokenizer, TEXT_COL, LABEL_COLS, MAX_LEN
        )
        val_dataset = HierarchicalHateDataset(
            val_df, tokenizer, TEXT_COL, LABEL_COLS, MAX_LEN
        )
        test_dataset = HierarchicalHateDataset(
            test_df, tokenizer, TEXT_COL, LABEL_COLS, MAX_LEN
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            drop_last=True
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False
        )

        test_loader = DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False
        )

        model = MultiHeadTransformerClassifier(
            model_name=model_name,
            num_labels_list=num_labels_list,
            dropout=0.3
        ).to(device)

        class_weights = compute_class_weights(
            train_df,
            LABEL_COLS,
            num_labels_dict
        )

        loss_fns = [
            nn.CrossEntropyLoss(weight=class_weights[i])
            for i in range(len(LABEL_COLS))
        ]

        optimizer = AdamW(
            model.parameters(),
            lr=LEARNING_RATE
        )

        total_steps = len(train_loader) * EPOCHS

        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(0.1 * total_steps),
            num_training_steps=total_steps
        )

        best_val_f1 = -1
        best_model_path = f"/content/best_{model_display_name.replace('/', '_')}.pt"

        for epoch in range(EPOCHS):
            print(f"\nEpoch {epoch+1}/{EPOCHS}")

            train_loss = train_one_epoch(
                model,
                train_loader,
                optimizer,
                scheduler,
                loss_fns
            )

            val_results, _, _ = evaluate_model(
                model,
                val_loader,
                loss_fns,
                LABEL_COLS
            )

            print("Train Loss:", train_loss)
            print("Validation Loss:", val_results["loss"])
            print("Validation Average Macro-F1:", val_results["average_macro_f1"])

            for col in LABEL_COLS:
                print(f"{col} Macro-F1: {val_results[col]['macro_f1']:.4f}")

            if val_results["average_macro_f1"] > best_val_f1:
                best_val_f1 = val_results["average_macro_f1"]
                torch.save(model.state_dict(), best_model_path)
                print("Best model saved.")

        # Load best model for testing
        model.load_state_dict(torch.load(best_model_path))

        test_results, test_true, test_pred = evaluate_model(
            model,
            test_loader,
            loss_fns,
            LABEL_COLS
        )

        all_model_results[model_display_name] = test_results

        print("\nTest Results for:", model_display_name)
        print("Average Macro-F1:", test_results["average_macro_f1"])

        for col in LABEL_COLS:
            print(f"{col}:")
            print("Accuracy:", test_results[col]["accuracy"])
            print("Macro Precision:", test_results[col]["macro_precision"])
            print("Macro Recall:", test_results[col]["macro_recall"])
            print("Macro F1:", test_results[col]["macro_f1"])

    except Exception as e:
        print(f"Error while training {model_display_name}: {e}")
        all_model_results[model_display_name] = "FAILED"


# ============================================================
# 15. CREATE BASELINE COMPARISON TABLE
# ============================================================

comparison_rows = []

for model_name, results in all_model_results.items():
    if results == "FAILED":
        row = {
            "Model": model_name,
            "Average Macro-F1": "FAILED"
        }

        for col in LABEL_COLS:
            row[f"{col} Macro-F1"] = "FAILED"

        comparison_rows.append(row)

    else:
        row = {
            "Model": model_name,
            "Average Macro-F1": results["average_macro_f1"]
        }

        for col in LABEL_COLS:
            row[f"{col} Accuracy"] = results[col]["accuracy"]
            row[f"{col} Macro-Precision"] = results[col]["macro_precision"]
            row[f"{col} Macro-Recall"] = results[col]["macro_recall"]
            row[f"{col} Macro-F1"] = results[col]["macro_f1"]

        comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)

comparison_df


# ============================================================
# 16. SAVE RESULTS TO EXCEL
# ============================================================

output_path = "/content/baseline_model_comparison_results.xlsx"
comparison_df.to_excel(output_path, index=False)

print("Saved baseline comparison results to:", output_path)


# ============================================================
# 17. SIMPLE MACRO-F1 SUMMARY TABLE
# ============================================================

summary_rows = []

for model_name, results in all_model_results.items():
    if results == "FAILED":
        continue

    row = {
        "Model": model_name,
        "L1 Macro-F1": results[LABEL_COLS[0]]["macro_f1"],
        "L2 Macro-F1": results[LABEL_COLS[1]]["macro_f1"],
        "L3 Macro-F1": results[LABEL_COLS[2]]["macro_f1"],
        "L4 Macro-F1": results[LABEL_COLS[3]]["macro_f1"],
        "L5 Macro-F1": results[LABEL_COLS[4]]["macro_f1"],
        "Average Macro-F1": results["average_macro_f1"]
    }

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(by="Average Macro-F1", ascending=False)

summary_df


# ============================================================
# 18. SAVE SUMMARY TABLE
# ============================================================

summary_path = "/content/baseline_macro_f1_summary.xlsx"
summary_df.to_excel(summary_path, index=False)

print("Saved summary results to:", summary_path)

Device: cuda


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


                                                Text Level1 - CLASS  \
0  S K  யாருடா நீ எனக்கே என்ன பாக்கனும் போல இருக்...           HATE   
1  டேய் உன்னோட சின்ன தங்கச்சி கொத்தும் கொலையுமா ந...           HATE   
2  பிரபல நாட்டாமை சுண்ணி ஊம்ப காசு அதிகம் வேணுமா?...           HATE   
3           அது ஒரு பொறம்போக்கு சிந்தனை கொண்ட ஜந்து.           HATE   
4   என்னோட சுன்னிய சப்பிட்டு தான் உன் குண்டியை கா...           HATE   

      Level 2 - TARGET Level 3 - CATEGORIES  \
0     Hate against Men             IMPLICIT   
1  Hate against LGBTQ+             EXPLICIT   
2  Hate against LGBTQ+             EXPLICIT   
3   No Target- Profane             EXPLICIT   
4  Hate against LGBTQ+             EXPLICIT   

                         Level 4 - SUBCATEGORIES Level 5 - BIAS Unnamed: 6  \
0  ImplicitMen-Undermining / Mocking Masculinity     Religional        NaN   
1                 ExplicitLGBTQ-Homophobic Slurs            NaN        NaN   
2                 ExplicitLGBTQ-Homophobic Slurs        

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

...

Training:   0%|          | 0/265 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/34 [00:00<?, ?it/s]

Train Loss: 8.412969277939707
Validation Loss: 8.762893396265367
Validation Average Macro-F1: 0.25224230239317086
Level1 - CLASS Macro-F1: 0.4123
Level 2 - TARGET Macro-F1: 0.2122
Level 3 - CATEGORIES Macro-F1: 0.4647
Level 4 - SUBCATEGORIES Macro-F1: 0.0364
Level 5 - BIAS Macro-F1: 0.1357
Best model saved.


Evaluating:   0%|          | 0/34 [00:00<?, ?it/s]


Test Results for: RoBERTa
Average Macro-F1: 0.2707038298216076
Level1 - CLASS:
Accuracy: 0.6522556390977443
Macro Precision: 0.43428386013131776
Macro Recall: 0.43403311145246626
Macro F1: 0.4337372297457828
Level 2 - TARGET:
Accuracy: 0.4774436090225564
Macro Precision: 0.2874304323881302
Macro Recall: 0.3047246468258141
Macro F1: 0.27034898916589345
Level 3 - CATEGORIES:
Accuracy: 0.5018796992481203
Macro Precision: 0.4872831961067255
Macro Recall: 0.468398704174058
Macro F1: 0.4707869423492657
Level 4 - SUBCATEGORIES:
Accuracy: 0.36466165413533835
Macro Precision: 0.0507978835978836
Macro Recall: 0.0618474215842637
Macro F1: 0.043426751637760816
Level 5 - BIAS:
Accuracy: 0.8984962406015038
Macro Precision: 0.12835660580021482
Macro Recall: 0.14285714285714285
Macro F1: 0.1352192362093352
Saved baseline comparison results to: /content/baseline_model_comparison_results.xlsx
Saved summary results to: /content/baseline_macro_f1_summary.xlsx
